In [ ]:
import json, pandas as pd

gt   = json.load(open("ground_truth_dx.json"))
m2   = json.load(open("predictions_model2_27b.json"))     # the 60 evaluated cases, we need the 60 cases keys. 
# ground_truth_dx.json holds 85 cases, from the earlier Model 1 4B run. Models 1 (27B), 2 and 3 were evaluated on 60 of them, so only those 60 cases’ diagnoses need labelling. 
# Any of the three 60-case prediction files gives the same 60 keys; I happened to pick Model 2’s.
uniq = sorted({x.strip() for c in m2 for x in gt[c]}, key=str.lower)
pd.DataFrame({"item": uniq, "inscope": ""}).to_csv("inscope_labels.csv", index=False)
print("unique items:", len(uniq))


# import json, pandas as pd

# gt   = json.load(open("ground_truth_dx.json"))
# CASES = list(json.load(open("predictions_model2_27b.json")).keys())   # the 60 evaluated case names only
# uniq = sorted({x.strip() for c in CASES for x in gt[c]}, key=str.lower)
# print("unique items:", len(uniq))
# # inscope_labels.csv is labelled and frozen: this cell no longer writes it

In [ ]:
v1 = pd.read_csv("inscope_labels_old.csv").fillna("")
v2 = pd.read_csv("inscope_labels.csv").fillna("")
m  = v1.merge(v2, on="item", suffixes=("_old", "_new"))
ch = m[m["inscope_old"].str.strip().str.upper() != m["inscope_new"].str.strip().str.upper()]
print("rows compared:", len(m), "| labels changed:", len(ch),
      "| N->Y:", (ch["inscope_new"].str.strip().str.upper() == "Y").sum(),
      "| Y->N:", (ch["inscope_new"].str.strip().str.upper() == "N").sum())

“In-scope labels were assigned by the author and reviewed by a domain expert without reference to model predictions. The final labels differ from the initial ones in 8 of 328 items (7 out of scope → in scope, 1 in scope → out of scope); all results use the final labels.”

In [ ]:
import json, pandas as pd

gt  = json.load(open("ground_truth_dx.json"))
lab = pd.read_csv("inscope_labels.csv").fillna("")
lab["inscope"] = lab["inscope"].str.strip().str.upper()
assert set(lab["inscope"]) <= {"Y", "N"}, "every item needs Y or N"
INSCOPE = set(lab.loc[lab["inscope"] == "Y", "item"].str.strip().str.lower())
print("in-scope items:", len(INSCOPE), "of", lab["item"].str.strip().str.lower().nunique())
JUDGES  = ["llama-3.1-8b-instruct", "granite-4.1-8b", "olmo-3-7b-instruct"]
SYSTEMS = {"Model 1": "model1_27b", "Model 2": "model2_27b", "Model 3": "model3_27b",
           "Model 3 + reasoning": "model3r_27b"}
KS = (1, 3, 5, 10, 15)

def recall(tag, k, inscope_only):
    V = [json.load(open(f"judge_v2_{tag}_{j}.json"))["verdicts"] for j in JUDGES]
    per_case = []
    for c in V[0]:
        keep = [i for i, x in enumerate(gt[c]) if not inscope_only or x.strip().lower() in INSCOPE]
        if not keep:
            continue                                   # no in-scope items: case skipped for every model
        found = [sum(any(v[c][i][:k]) for v in V) / len(V) for i in keep]   # judges averaged per item
        per_case.append(sum(found) / len(found))
    return round(sum(per_case) / len(per_case), 3), len(per_case)



def ceiling(k, inscope_only):
    """Best possible recall@k: a perfect model finds min(k, n) of a case's n items."""
    vals = []
    for c in json.load(open(f"judge_v2_model1_27b_{JUDGES[0]}.json"))["verdicts"]:
        n = sum(1 for x in gt[c] if not inscope_only or x.strip().lower() in INSCOPE)
        if n:
            vals.append(min(k, n) / n)
    return round(sum(vals) / len(vals), 3), len(vals)

rows = []
for name, tag in SYSTEMS.items():
    for scope in (True, False):
        r = {"system": name, "scope": "in-scope" if scope else "full"}
        for k in KS:
            r[f"@{k}"], r["cases"] = recall(tag, k, scope)
        rows.append(r)
for scope in (True, False):
    r = {"system": "Ceiling (perfect model)", "scope": "in-scope" if scope else "full"}
    for k in KS:
        r[f"@{k}"], r["cases"] = ceiling(k, scope)
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

* In-scope: Model 3 is best, finding 67% of correct items, against about 59–60% for the others.
* Full set: Model 1 is best (57%) and Model 3 drops to third (54%), so its advantage only holds for in-scope items.
* Reasoning: adding it to Model 3 lowers recall in both settings.
* Room to improve: a perfect model would find nearly all items, so even the best system misses about a third (in-scope) to over 40% (full).

In [ ]:
def per_case(tag, k, inscope_only):
    """Per-case recall@k (judges averaged per item), keyed by case ID.
    Same logic as recall(), but returns the individual case values."""
    V = [json.load(open(f"judge_v2_{tag}_{j}.json"))["verdicts"] for j in JUDGES]
    out = {}
    for c in V[0]:
        keep = [i for i, x in enumerate(gt[c]) if not inscope_only or x.strip().lower() in INSCOPE]
        if not keep:
            continue
        found = [sum(any(v[c][i][:k]) for v in V) / len(V) for i in keep]
        out[c] = sum(found) / len(found)
    return out

In [ ]:
import numpy as np

rng = np.random.default_rng(1234)

def boot_ci(x, n=10_000):
    x = np.asarray(x, dtype=float)
    means = [rng.choice(x, size=len(x), replace=True).mean() for _ in range(n)]
    return x.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

for scope in ("in-scope", "full"):
    print(f"\nRecall@15, {scope}")
    for name, tag in SYSTEMS.items():
        x = per_case(tag, 15, scope == "in-scope")      # ← same call as in your PAIRS loop
        x = list(x.values()) if isinstance(x, dict) else x
        m, lo, hi = boot_ci(x)
        print(f"  {name:20s} {m:.3f}  (95% CI {lo:.3f} to {hi:.3f})   n = {len(x)}")

* In-scope: Model 3 still has the highest point estimate (0.670). Its interval (0.604–0.733) overlaps all the others, so the advantage is suggestive but not clearly significant.
* Full set: all four systems sit between 0.52 and 0.57 with heavily overlapping intervals. They are effectively tied, so Model 1’s lead isn’t meaningful.
* Reasoning: the drop it causes (0.670 → 0.598 in-scope) is also within overlapping intervals, so you can’t claim it reliably hurts, only that it doesn’t help.
* Caveat: overlapping intervals don’t rule out a real difference, because the models are tested on the same cases. A paired bootstrap on the difference (e.g. Model 3 minus Model 1, per case) is the right test and may show the in-scope gap is significant.
* Bottom line: with about 60 cases, no system is clearly better at Recall@15. Model 3 in-scope is the most promising result to confirm with a paired test.

In [ ]:
import re

lab_now = pd.read_csv("inscope_labels.csv").fillna("")
LABEL = dict(zip(lab_now["item"].str.strip().str.lower(), lab_now["inscope"].str.strip().str.upper()))

for c in json.load(open(f"judge_v2_model1_27b_{JUDGES[0]}.json"))["verdicts"]:
    if any(x.strip().lower() in INSCOPE for x in gt[c]):
        continue
    print("CASE:", c, "| ground-truth items after cleaning:", len(gt[c]))
    for x in gt[c]:
        print("   ", LABEL.get(x.strip().lower(), "?"), "|", x)

    # the raw Discharge Diagnosis section, before cleaning
    text = open(f"patient_targets/{c.replace('.txt', '_target.txt')}", encoding="utf-8").read()
    m = re.search(r"Discharge Diagnos[ie]s\s*:(.*?)(?=\n\s*(?:Discharge Condition|Discharge Instructions|Condition at Discharge|Mental Status|Followup|Follow-up)\b|\Z)",
                  text, re.I | re.S)
    print("\nRAW DISCHARGE DIAGNOSIS SECTION:\n", m.group(1).strip() if m else "(not found)")

In [ ]:
import random

def per_case(tag, k, inscope_only):
    V = [json.load(open(f"judge_v2_{tag}_{j}.json"))["verdicts"] for j in JUDGES]
    out = {}
    for c in V[0]:
        keep = [i for i, x in enumerate(gt[c]) if not inscope_only or x.strip().lower() in INSCOPE]
        if keep:
            out[c] = sum(sum(any(v[c][i][:k]) for v in V) / len(V) for i in keep) / len(keep)
    return out

def paired(a, b, k, inscope_only, n=10000, seed=1234):
    A, B = per_case(SYSTEMS[a], k, inscope_only), per_case(SYSTEMS[b], k, inscope_only)
    d = [A[c] - B[c] for c in sorted(A)]
    mean = sum(d) / len(d)
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(d) for _ in d) / len(d) for _ in range(n))                 # 95% CI
    extreme = sum(abs(sum(x if rng.random() < 0.5 else -x for x in d) / len(d)) >= abs(mean) - 1e-12
                  for _ in range(n))                                                          # permutation test
    return round(mean, 3), round(boots[int(0.025 * n)], 3), round(boots[int(0.975 * n) - 1], 3), (extreme + 1) / (n + 1)

def holm(p):
    order, adj, run = sorted(range(len(p)), key=lambda i: p[i]), [0] * len(p), 0
    for rank, i in enumerate(order):
        run = max(run, min(1, (len(p) - rank) * p[i]))
        adj[i] = run
    return adj

PAIRS = [("Model 2", "Model 1"), ("Model 3", "Model 2"), ("Model 3", "Model 1")]
for label, scope, k in [("PRIMARY: in-scope @15", True, 15),
                        ("secondary: in-scope @5", True, 5),
                        ("secondary: full @15", False, 15)]:
    res = [paired(a, b, k, scope) for a, b in PAIRS]
    adj = holm([r[3] for r in res])
    print("\n" + label)
    for (a, b), (m, lo, hi, p), ph in zip(PAIRS, res, adj):
        print(f"  {a} - {b}: diff {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}) | p={p:.4f} | Holm p={ph:.4f}")

print("\nSEPARATE TEST (not in the Holm family): Model 3 + reasoning - Model 3")
for label, scope, k in [("PRIMARY: in-scope @15", True, 15), ("in-scope @5", True, 5), ("full @15", False, 15)]:
    m, lo, hi, p = paired("Model 3 + reasoning", "Model 3", k, scope)
    print(f"  {label}: diff {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}) | p={p:.4f}")

* Model 3 vs Models 1 and 2 (primary, in-scope @15): Model 3 is about 8 points better than both, and the uncorrected intervals exclude zero (p ≈ 0.03). After Holm correction, p = 0.083, so the gain is a consistent trend but not statistically significant at 0.05.
* Model 2 vs Model 1: no difference at any setting (diff ≈ 0, p > 0.5), so Model 2 adds nothing over Model 1.
* @5 (secondary): same pattern. Model 3 leads by 6–8 points but nothing survives correction (best Holm p = 0.066), so the trend holds at a shallower cutoff.
* Full set: no differences at all (all Holm p = 1.0). Model 3’s advantage is confined to in-scope items.
* Reasoning: adding it to Model 3 significantly lowers in-scope recall: −7.2 points at @15 (p = 0.001) and −5.6 points at @5 (p = 0.02). This is your clearest result, and the @15 effect would survive any reasonable correction. On the full set the drop is small and not significant.

* Bottom line: Model 3 shows a consistent but not significant in-scope improvement, and reasoning reliably hurts it.

In [ ]:
def krippendorff_alpha_nominal(units):
    """units: one list of 0/1 ratings per item (one rating per judge)."""
    o = {(0, 0): 0.0, (0, 1): 0.0, (1, 0): 0.0, (1, 1): 0.0}
    for r in units:
        m = len(r)
        for a in range(m):
            for b in range(m):
                if a != b:
                    o[(r[a], r[b])] += 1.0 / (m - 1)
    n0, n1 = o[(0, 0)] + o[(0, 1)], o[(1, 0)] + o[(1, 1)]
    n = n0 + n1
    if n0 == 0 or n1 == 0:
        return float("nan")
    return 1 - ((o[(0, 1)] + o[(1, 0)]) / n) / (2 * n0 * n1 / (n * (n - 1)))

rows = []
for name, tag in SYSTEMS.items():
    V = [json.load(open(f"judge_v2_{tag}_{j}.json"))["verdicts"] for j in JUDGES]
    for scope in (True, False):
        r = {"system": name, "scope": "in-scope" if scope else "full"}
        for k in (5, 15):
            units = [[int(any(v[c][i][:k])) for v in V]
                     for c in V[0] for i, x in enumerate(gt[c])
                     if not scope or x.strip().lower() in INSCOPE]
            r[f"alpha@{k}"] = round(krippendorff_alpha_nominal(units), 3)
            r[f"agree@{k}"] = round(sum(len(set(u)) == 1 for u in units) / len(units), 3)
        r["items"] = len(units)
        rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

* Reliability is acceptable throughout. Alpha ranges from 0.72 to 0.85 and raw agreement from 79% to 91%. Every value is above Krippendorff’s 0.667 floor for tentative conclusions, so the relevance labels behind your recall scores are usable.
* @5 is more reliable than @15. At @5, alpha is mostly around 0.80 or higher, which is the “reliable” threshold. At @15 it drops to 0.72–0.81, likely because deeper results include more borderline items that raters judge differently. Your primary @15 results therefore rest on somewhat less certain labels.
* Full is slightly more reliable than in-scope. Alpha is about 0.02–0.07 higher on the full set, so in-scope items appear harder to judge consistently.
* Model 1 gets the most consistent judgments and Model 2 the least. The gap is small (about 0.05) and doesn’t change the overall picture. Model 3 sits between the two, and adding reasoning doesn’t change its reliability.
* Bottom line: label quality is good enough to support your findings. For @15 in-scope, where alpha is about 0.72–0.76, note that the labels are moderately rather than highly reliable.

In [ ]:
EJ = [json.load(open(f"expl_v1_model3r_27b_{j}.json"))["items"] for j in JUDGES]
R  = [{(r["case"], r["rank"]): r for r in e} for e in EJ]
keys = sorted(R[0])

rows = []
for q in ("specific", "supportive", "both"):
    val = (lambda r: r["specific"] and r["supportive"]) if q == "both" else (lambda r, q=q: r[q])
    units = [[int(val(Rj[key])) for Rj in R] for key in keys]
    rows.append({"question": q,
                 "rate (judges averaged)": round(sum(sum(u) / len(u) for u in units) / len(units), 3),
                 "rate (majority)": round(sum(sum(u) >= 2 for u in units) / len(units), 3),
                 "raw agreement": round(sum(len(set(u)) == 1 for u in units) / len(units), 3),
                 "alpha": round(krippendorff_alpha_nominal(units), 3),
                 "items": len(units)})
print(pd.DataFrame(rows).to_string(index=False))

for j, Rj in zip(JUDGES, R):
    print(f"  {j:24} specific {sum(r['specific'] for r in Rj.values())/len(Rj):.3f} | "
          f"supportive {sum(r['supportive'] for r in Rj.values())/len(Rj):.3f}")

* The judges rate almost everything positively. About 93–96% of items are judged specific and supportive, and about 89–90% are judged both. The averaged and majority-vote rates are nearly identical, so these aggregate figures are stable.
* High agreement but low alpha (≈ 0.41–0.43) is the prevalence effect. When nearly every item is a “yes”, chance agreement is already very high, so the 83–91% raw agreement adds little beyond that and alpha drops. The judges agree on the easy majority but not on the few borderline items.
* Alpha is below the 0.667 threshold. Item-level judgments are not reliable on their own. Use these as aggregate rates, not per-item labels.
* The judges differ in strictness. Granite is the most lenient (about 97% on both questions). OLMo is strictest on specific (88%) and Llama strictest on supportive (90%), so each judge disagrees on a different dimension. That pattern is likely what’s driving the low alpha.
* Bottom line: outputs are judged overwhelmingly specific and supportive, but the judges may be too lenient to tell systems apart. Validate a small subsample with human annotation before putting much weight on these numbers.

In [ ]:
import re, random
from pathlib import Path

LINE_RE = re.compile(r"^\s*\d+[.)]\s*(.+?)\s*\|\s*(.+?)\s*$")
P3R = json.load(open("predictions_model3r_27b.json"))
def top5(raw): return [m.groups() for l in raw.splitlines() if (m := LINE_RE.match(l))][:5]

sample = random.Random(42).sample(keys, 20)                 # fixed seed: reproducible choice
if not Path("expl_human_check.csv").exists():
    pd.DataFrame([{"case": c, "rank": r,
                   "diagnosis": top5(P3R[c]["raw"])[r - 1][0],
                   "justification": top5(P3R[c]["raw"])[r - 1][1],
                   "specific": "", "supportive": ""} for c, r in sample]
                ).to_csv("expl_human_check.csv", index=False)
print("expl_human_check.csv ready:", len(sample), "items")

In [ ]:
h = pd.read_csv("expl_human_check.csv").fillna("")
for q in ("specific", "supportive"):
    h[q] = h[q].str.strip().str.upper()
assert set(h["specific"]) | set(h["supportive"]) <= {"Y", "N"}, "every row needs Y or N in both columns"

agree = {"specific": 0, "supportive": 0, "both": 0}
for _, r in h.iterrows():
    js = [Rj[(r["case"], int(r["rank"]))] for Rj in R]                     # the 3 judges' answers
    maj = {"specific":   sum(x["specific"] for x in js) >= 2,
           "supportive": sum(x["supportive"] for x in js) >= 2,
           "both":       sum(x["specific"] and x["supportive"] for x in js) >= 2}
    you = {"specific": r["specific"] == "Y", "supportive": r["supportive"] == "Y"}
    you["both"] = you["specific"] and you["supportive"]
    for q in agree:
        agree[q] += you[q] == maj[q]

print("agreement with the judges' majority:", {q: f"{n}/{len(h)}" for q, n in agree.items()})
print("your rates:", {q: f"{(h[q] == 'Y').mean():.2f}" for q in ("specific", "supportive")})

In [ ]:
from sklearn.metrics import cohen_kappa_score

# expert's answers and the judges' majority, one list each
expert = {"specific": [], "supportive": [], "both": []}
judges = {"specific": [], "supportive": [], "both": []}

for _, r in h.iterrows():
    js = [Rj[(r["case"], int(r["rank"]))] for Rj in R]
    spec, supp = r["specific"] == "Y", r["supportive"] == "Y"
    expert["specific"].append(spec)
    expert["supportive"].append(supp)
    expert["both"].append(spec and supp)
    judges["specific"].append(sum(x["specific"] for x in js) >= 2)
    judges["supportive"].append(sum(x["supportive"] for x in js) >= 2)
    judges["both"].append(sum(x["specific"] and x["supportive"] for x in js) >= 2)

for q in ("specific", "supportive", "both"):
    k = cohen_kappa_score(expert[q], judges[q])
    print(f"{q:10s}  Cohen's kappa = {k:.2f}   "
          f"(expert yes: {sum(expert[q])}/20, judges yes: {sum(judges[q])}/20)")

* Specific (κ = −0.09): no agreement with the expert. The judges say yes to 19 of 20 items, while the expert says yes to only 14. They miss almost all the items the expert considers non-specific, so the LLM judges cannot be trusted on specificity.
* Supportive (κ = 0.46): moderate agreement. The judges and expert are closer (19 vs 17 yes). This is the only dimension where the judges are reasonably usable, though still imperfect.
* Both (κ = 0.12): slight agreement. The poor specificity judgments drag this down.
* This confirms the leniency concern. The judges overrate outputs: the expert finds about 70% of items specific, against the judges’ 93%. The high rates in the previous table are therefore inflated, especially for specificity.
* Caveat: with only 20 items these kappas are imprecise, but the direction is clear. In the write-up, present supportive results with caution, treat specificity results as unreliable, and report the expert-validated rates alongside the judge rates.

In [ ]:
dirs = {"judges Y, you N": 0, "judges N, you Y": 0}
for _, r in h.iterrows():
    js = [Rj[(r["case"], int(r["rank"]))] for Rj in R]
    maj = sum(x["specific"] for x in js) >= 2
    you = r["specific"] == "Y"
    if maj and not you: dirs["judges Y, you N"] += 1
    if you and not maj: dirs["judges N, you Y"] += 1
print("Specific disagreements:", dirs)
print("your 'both' rate:", ((h["specific"] == "Y") & (h["supportive"] == "Y")).mean())

In [ ]:
import random

#case  = random.Random(42).choice(sorted(json.load(open(f"judge_v2_model1_27b_{JUDGES[0]}.json"))["verdicts"]))
cases = sorted(json.load(open(f"judge_v2_model1_27b_{JUDGES[0]}.json"))["verdicts"])
case  = cases[5]          # change 5 to any number from 0 to 59


preds = {name: json.load(open(f"predictions_{tag}.json"))[case]["diagnoses"][:15] for name, tag in SYSTEMS.items()}
V     = {tag: [json.load(open(f"judge_v2_{tag}_{j}.json"))["verdicts"][case] for j in JUDGES] for tag in SYSTEMS.values()}

def found(tag, i):                      # found in the top 15 by at least 2 of the 3 judges
    return sum(any(v[i][:15]) for v in V[tag]) >= 2

print("GROUND TRUTH   (* = in scope)                   found by:   M1  M2  M3  M3+R")
for i, x in enumerate(gt[case]):
    marks = "   ".join("✓" if found(tag, i) else "·" for tag in SYSTEMS.values())
    print(f" {'*' if x.strip() in INSCOPE else ' '} {x[:55]:<55}  {marks}")

for name, p in preds.items():
    print(f"\n{name} (top 15):")
    for r, d in enumerate(p, 1):
        print(f"  {r:>2}. {d}")

In [ ]:
import json, re, collections
import pandas as pd

LIST_RE = re.compile(r"^\s*(?:\d+[.)]|[-*\u2022])\s*(.+?)\s*$")
CAP = 15
PRED_FILES = {                                    # adjust names to match SYSTEMS
    "M1":   "predictions_model1_27b.json",
    "M2":   "predictions_model2_27b.json",
    "M3":   "predictions_model3_27b.json",
    "M3+R": "predictions_model3r_27b.json",
}

rows = []
for name, path in PRED_FILES.items():
    P = json.load(open(path, encoding="utf-8"))
    for k, v in P.items():
        lines = [l for l in v["raw"].splitlines() if l.strip()]
        other = [l for l in lines if not LIST_RE.match(l) and l.strip().lower().rstrip(".") != "none"]
        n = len(v["diagnoses"])
        rows.append(dict(system=name, case=k, n_returned=n, n_scored=min(n, CAP),
                         truncated=v["finish"] == "length", parse_fail=(n == 0),
                         other_lines=len(other),
                         no_reason=sum(1 for j in v.get("justifications", []) if not j) if name == "M3+R" else None))
df = pd.DataFrame(rows)

summary = df.groupby("system", sort=False).agg(
    cases=("case", "count"),
    mean_scored=("n_scored", "mean"),
    min_scored=("n_scored", "min"),
    cases_under_15=("n_scored", lambda s: int((s < CAP).sum())),
    truncated=("truncated", "sum"),
    parse_fail=("parse_fail", "sum"),
    cases_with_other_lines=("other_lines", lambda s: int((s > 0).sum())),
).round(2)
print(summary.to_string())
print("\nM3+R diagnoses without a justification:", int(df.loc[df.system == "M3+R", "no_reason"].sum()))
print("\nDistribution of n_scored per system:")
print(pd.crosstab(df.system, df.n_scored).to_string())

In [ ]:
import json, statistics as st

for name, tag in SYSTEMS.items():
    P = json.load(open(f"predictions_{tag}.json"))
    n_items, capped, empty = [], 0, 0
    for case, v in P.items():
        dx = v["diagnoses"] if isinstance(v, dict) else v
        n_items.append(len(dx))
        if len(dx) == 0:
            empty += 1                       # nothing could be parsed
        if isinstance(v, dict) and v.get("finish") == "length":
            capped += 1                      # hit the 600-token limit
    print(f"{name:20s} median items {st.median(n_items):4.1f} | "
          f"min {min(n_items):2d} | <15 items: {sum(n < 15 for n in n_items):2d}/60 | "
          f"hit token cap: {capped:2d} | parse failures: {empty}")